# 06. Inferencja (Projekcja) i Analiza MAE Przygotowana ściśle pod zbiór Testowy


In [1]:
%load_ext autoreload
%autoreload 2

import torch
from torch.utils.data import Dataset, DataLoader
import polars as pl
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
from sklearn.metrics import mean_absolute_error
import gc
import warnings
import joblib
warnings.filterwarnings('ignore')

from models import TimeSeriesLSTM, TimeSeriesTransformer, get_device

SEQ_LEN = 36
BATCH_SIZE = 2048

# ZUNIFIKOWANE ODCINANIE TYPÓW ZE STANDARDU TRENINGOWEGO
preproc_state = joblib.load('preprocessor_fitted.joblib')
FEATURE_COLS = preproc_state.numeric_cols_

## 1. Zaczep C-Array dla Zbioru Testowego (`Test Dataset`)

In [2]:
def create_memmaps(csv_path, x_path, y_path, feature_cols, target_col='x2'):
    if os.path.exists(x_path) and os.path.exists(y_path):
        print(f"Macierze binarne {x_path} i {y_path} już istnieją!\n")
        return
        
    print("1/3 Liczenie obciążenia ramki... (Test CSV)")
    total_rows = pl.scan_csv(csv_path).select(pl.len()).collect().item()
    
    print(f"2/3 Alokacja pod {total_rows} rzędów na wirtualnej macierzy dyskowej...")
    X_mm = np.memmap(x_path, dtype='float32', mode='w+', shape=(total_rows, len(feature_cols)))
    y_mm = np.memmap(y_path, dtype='float32', mode='w+', shape=(total_rows,))
    
    print("3/3 Grawerowanie chunkami (Zabezpieczenie przed brakiem targetu)... ")
    chunksize = 200000
    start_idx = 0
    
    for chunk in tqdm(pd.read_csv(csv_path, chunksize=chunksize, low_memory=False), 
                      total=(total_rows//chunksize) + 1, desc="Konwersja Binarna (TEST)"):
        
        # Selekcja na wypadki brakujących zmiennych w środowisku obcym
        valid_feats = [c for c in feature_cols if c in chunk.columns]
        assert len(valid_feats) == len(feature_cols), f"Krytyczny błąd: W paczce danych brakuje wymaganych kolumn cech! Znaleziono {len(valid_feats)} z {len(feature_cols)}."
        X_chunk = chunk[valid_feats].fillna(0.0).values.astype(np.float32)
        X_mm[start_idx:start_idx+len(chunk)] = X_chunk
        
        # Jeśli przewidywany Blind Test nie ma etykiety x2, sztucznie utrzymujemy wektor 
        if target_col in chunk.columns:
            y_chunk = chunk[target_col].fillna(0.0).values.astype(np.float32)
        else:
            y_chunk = np.zeros(len(chunk), dtype=np.float32)
            
        y_mm[start_idx:start_idx+len(chunk)] = y_chunk
        
        start_idx += len(chunk)
        X_mm.flush() 
        y_mm.flush()
        
    print("Eksport Test-Wektorów pomyślnie scalony !!!")

# Wykonanie konwersji absolutnie na docelowym pliku TESTOWYM!
create_memmaps('data/transformed_data_test.csv', 'data/data_test_X.npy', 'data/data_test_y.npy', FEATURE_COLS)

class MemmapInferenceDataset(Dataset):
    def __init__(self, csv_path, x_path, y_path, feature_cols, seq_len=36, target_col='x2'):
        df_devs = pl.read_csv(csv_path, columns=['deviceId'])
        total_rows = len(df_devs)

        # Wyizolowane, w 100% bezpieczne blokowe sekwencjonowanie pliku CSV chroniące spójność indexów
        dev_array = df_devs.to_series().to_numpy()
        changes = np.where(dev_array[:-1] != dev_array[1:])[0] + 1
        device_counts = np.diff(np.concatenate(([0], changes, [total_rows])))
        
        assert len(dev_array) == 0 or len(np.unique(dev_array)) == len(changes) + 1, "Krytyczny błąd: Urządzenia nie tworzą spójnych bloków!"
        
        del df_devs
        gc.collect()
        
        self.feature_cols = feature_cols
        
        self.X = np.memmap(x_path, dtype='float32', mode='r', shape=(total_rows, len(self.feature_cols)))
        self.y = np.memmap(y_path, dtype='float32', mode='r', shape=(total_rows,))
        
        self.seq_len = seq_len
        self.indices = []
        current_idx = 0
        for count in tqdm(device_counts, desc="Synchronizacja Liniowa Osnowy Testowej"):
            if count >= self.seq_len:
                valid_starts = np.arange(current_idx, current_idx + count - self.seq_len + 1)
                self.indices.extend(valid_starts)
            current_idx += count

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start = self.indices[idx]
        end = start + self.seq_len
        x_seq = torch.tensor(np.array(self.X[start:end]), dtype=torch.float32)
        target = torch.tensor(self.y[end - 1], dtype=torch.float32)
        return x_seq, target

# Wczytywanie macierzy przygotowanych dla procesu TEST!
inf_dataset = MemmapInferenceDataset('data/transformed_data_test.csv', 'data/data_test_X.npy', 'data/data_test_y.npy', FEATURE_COLS, seq_len=SEQ_LEN)
inf_loader = DataLoader(inf_dataset, batch_size=BATCH_SIZE, shuffle=False)


Macierze binarne data/data_test_X.npy i data/data_test_y.npy już istnieją!



Synchronizacja Liniowa Osnowy Testowej: 100%|██████████| 595/595 [00:00<00:00, 2984.43it/s]


## 2. Inicjalizacja Sieci z Pliku Zapisów `.pth`

In [3]:
device = get_device()
INPUT_SIZE = len(inf_dataset.feature_cols)

lstm_net = TimeSeriesLSTM(input_size=INPUT_SIZE, hidden_size=128, num_layers=2, output_size=1, dropout=0.2)
lstm_net.load_state_dict(torch.load('saved_models/model_lstm.pth', map_location=device))
lstm_net.to(device)
lstm_net.eval()

transformer_net = TimeSeriesTransformer(
    input_size=INPUT_SIZE, d_model=128, nhead=4, num_layers=2, 
    output_size=1, dim_feedforward=512, dropout=0.1, max_seq_len=SEQ_LEN
)
transformer_net.load_state_dict(torch.load('saved_models/model_transformer.pth', map_location=device))
transformer_net.to(device)
transformer_net.eval()
print("Modele wybudzone i rzutowane na pancerne Inferowanie Testowe.")

Modele wybudzone i rzutowane na pancerne Inferowanie Testowe.


## 3. Emisja Ocen Architektury (Przejście Tensorów)

In [4]:
def predict(model, loader, model_name):
    preds = []
    with torch.no_grad():
        for X_batch, _ in tqdm(loader, desc=f"{model_name}"):
            X_batch = X_batch.to(device)
            outputs = model(X_batch).cpu().numpy().squeeze()
            preds.extend(outputs)
    return np.array(preds)

lstm_preds = predict(lstm_net, inf_loader, "LSTM - Mmap Scan")
trf_preds = predict(transformer_net, inf_loader, "TRANSFORMER - Mmap Scan")

# Ściśle klarowna struktura odczytywania faktycznych indeksów do ewaluacji
valid_indices = [idx + SEQ_LEN - 1 for idx in inf_dataset.indices]
trues = np.array([inf_dataset.y[true_idx] for true_idx in valid_indices]).squeeze()


TRANSFORMER - Mmap Scan: 100%|██████████| 2409/2409 [04:40<00:00,  8.57it/s]


## 4. Analitka Ocenowa z OOM protection (Metryki Skuteczności Błędu)

In [5]:
del lstm_net
del transformer_net
gc.collect()

# PANCERNY odczyt DataFrame przy pomocy Numpy Mask zabezpieczający przez załamaniem Polars slices
mask = np.zeros(len(inf_dataset.X), dtype=bool)
mask[valid_indices] = True
results_df = pl.read_csv('data/transformed_data_test.csv', columns=['deviceId', 'timedate', 'x2']).filter(pl.Series(mask))

assert np.allclose(results_df['x2'].fill_null(0.0).to_numpy(), trues, atol=1e-5), "Krytyczny błąd: Rozbieżność w pozycjonowaniu wierszy maski względem Memmapy!"
results_df = results_df.with_columns([
    pl.Series('x2_true', trues),
    pl.Series('x2_pred_lstm', lstm_preds),
    pl.Series('x2_pred_trf', trf_preds),
])

assert np.allclose(results_df['x2'].fill_null(0.0).to_numpy(), trues, atol=1e-5), "Krytyczny błąd: Rozbieżność w pozycjonowaniu wierszy maski względem Memmapy!"
results_df = results_df.with_columns([
    pl.col('timedate').str.slice(0, 7).alias('month')
])

monthly_agg = results_df.group_by(['deviceId', 'month']).agg([
    pl.col('x2_true').mean().alias('monthly_true_x2'),
    pl.col('x2_pred_lstm').mean().alias('monthly_pred_lstm'),
    pl.col('x2_pred_trf').mean().alias('monthly_pred_trf')
])

# Na wypadek pustej puli testowej (Zera) minimalizujemy kolizje metryk
mae_lstm = mean_absolute_error(monthly_agg['monthly_true_x2'].to_numpy(), monthly_agg['monthly_pred_lstm'].to_numpy())
mae_trf = mean_absolute_error(monthly_agg['monthly_true_x2'].to_numpy(), monthly_agg['monthly_pred_trf'].to_numpy())

display(monthly_agg.head(10))
print("\n==============================================================")
print(f"⭐ Odchylenie MAE ZBIORU TESTOWEGO (Miesięczny per Stacja) LSTM:         {mae_lstm:.4f}")
print(f"⭐ Odchylenie MAE ZBIORU TESTOWEGO (Miesięczny per Stacja) TRANSFORMER:   {mae_trf:.4f}")
print("==============================================================")


deviceId,month,monthly_true_x2,monthly_pred_lstm,monthly_pred_trf
str,str,f32,f32,f32
"""46ef8778e229b5a977dfa1a24b1707…","""2025-04""",0.055713,0.056164,0.054673
"""17ca3943eb703954ecd459dde27cad…","""2025-04""",0.025576,0.020242,0.017278
"""d3b2c314fb99dea50f2097328448a3…","""2025-04""",0.011106,0.009573,0.005967
"""60c5b163f2f9c8555febf4fda3ecf8…","""2025-04""",0.051052,0.054801,0.052523
"""f56d840a338b79cf5a16081fc0195b…","""2025-04""",0.122232,0.124881,0.129269
"""f99b61a31667174a9d7b6dbc01be69…","""2025-04""",0.032076,0.032021,0.031088
"""a7c6b26faf3eec610f35111b1c22a6…","""2025-04""",0.082411,0.086979,0.082506
"""16654eae2d44b2e4d12caa7a530ed6…","""2025-04""",0.276868,0.279872,0.297787
"""fbdf266dd372334fcd865972f98057…","""2025-04""",0.049685,0.053238,0.052206



⭐ Odchylenie MAE ZBIORU TESTOWEGO (Miesięczny per Stacja) LSTM:         0.0023
⭐ Odchylenie MAE ZBIORU TESTOWEGO (Miesięczny per Stacja) TRANSFORMER:   0.0046


In [6]:
results_df.write_csv('data/predictions_results.csv')
print("Predykcje zapisane do data/predictions_results.csv")

Predykcje zapisane do data/predictions_results.csv
